<a href="https://colab.research.google.com/github/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-1-probabilidad-y-variables-aleatorias/u2_ejemplos_variable_aleatoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>


# Unidad 2 — Variable aleatoria, en ejemplos

**Asignatura E8102 · Exploración de datos**

Este notebook acompaña al apunte de la unidad 2. No agrega teoría: toma los dos ejemplos del apunte,
el bolillero sin reposición y las personas por hogar del relevamiento, y deja mover cada cosa que el
apunte calcula a mano. Cada bloque dice a qué sección corresponde y qué conviene mirar.

Se lee el apunte primero y se juega con esto después.


## Cómo se usa

Antes de tocar nada: `Archivo` → `Guardar una copia en Drive`. Si no, los cambios se pierden.

Este notebook no pide escribir código. Cada bloque tiene un **formulario** con deslizadores y
listas desplegables. Se mueve un valor y el gráfico se vuelve a dibujar solo.

Tres pasos:

1. Ejecutar todo una vez: `Entorno de ejecución` → `Ejecutar todo`. Tarda unos segundos y deja
   listas las herramientas y los datos.
2. Bajar hasta un bloque, leer el resumen y mover los parámetros del formulario. La celda se
   ejecuta sola con cada cambio.
3. Si un formulario no reacciona, hacer clic adentro y apretar `Shift + Enter`, o volver al paso 1.

El código de cada formulario está oculto a propósito. Si te da curiosidad, `Mostrar código` lo
despliega, pero nada de este notebook requiere leerlo.


## Preparación

Dos celdas. La primera carga las herramientas y la paleta de colores. La segunda carga el
relevamiento de hogares del T8001, que los dos apuntes usan como ejemplo con datos reales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import Circle, Polygon, Rectangle
from cycler import cycler
from fractions import Fraction
from io import StringIO
import scipy
from scipy import stats

# La paleta de las ilustraciones del apunte, para que los gráficos se vean como el mismo material.
CREMA, TEAL, TEAL_PALIDO, MOSTAZA, CORAL, TINTA = (
    "#FEF8EF", "#61A9A1", "#D0E5DE", "#EEA92F", "#F07C52", "#32435D")

plt.rcParams.update({
    "figure.facecolor": CREMA, "axes.facecolor": CREMA, "savefig.facecolor": CREMA,
    "axes.edgecolor": TINTA, "axes.labelcolor": TINTA, "text.color": TINTA,
    "xtick.color": TINTA, "ytick.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": cycler(color=[TEAL, CORAL, MOSTAZA, TINTA]),
    "axes.grid": True, "grid.color": TEAL_PALIDO, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "figure.figsize": (9, 4.5), "figure.dpi": 100,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "font.size": 11,
})


def num(x, decimales=4):
    """Número con coma decimal, como en el apunte."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    if abs(x) < 0.5 * 10 ** -decimales:
        x = 0.0   # evita el "-0,0000"
    return f"{x:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def frac(k, n):
    """'k/n = 0,xxxx' con la fracción sin simplificar, que es como la escribe el apunte."""
    return f"{k}/{n} = {num(k / n)}"


print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__, "| todo listo")


def medidas(valores, probabilidades):
    """Las seis medidas de la unidad, a partir de una distribución: valores y sus probabilidades.

    Es la distribución completa: la varianza divide por el total, sin corrección muestral."""
    x = np.asarray(valores, dtype=float)
    p = np.asarray(probabilidades, dtype=float)
    p = p / p.sum()
    mu = (x * p).sum()
    var = (((x - mu) ** 2) * p).sum()
    sd = np.sqrt(var)
    z = (x - mu) / sd if sd > 0 else np.zeros_like(x)
    return {
        "E(X)": mu, "V(X)": var, "desvío standard": sd,
        "CV (%)": 100 * sd / mu if mu else np.nan,
        "asimetría α₃": (z ** 3 * p).sum() if sd > 0 else np.nan,
        "kurtosis α₄": (z ** 4 * p).sum() if sd > 0 else np.nan,
    }


def distribucion_empirica(serie):
    """Valores distintos, su probabilidad (frecuencia relativa) y su cantidad."""
    conteo = serie.value_counts().sort_index()
    return conteo.index.to_numpy(), (conteo / conteo.sum()).to_numpy(), conteo.to_numpy()


def tabla_medidas(*columnas, decimales=4):
    """Una tabla con una columna por distribución. columnas = (nombre, dict de medidas)."""
    df = pd.DataFrame({nombre: pd.Series(m) for nombre, m in columnas})
    return df.map(lambda v: num(v, 2 if abs(v) >= 100 else decimales))


def dibujar_fulcro(ax, x, color=CORAL, relleno=True, tamano=15):
    """El punto de apoyo de la esperanza: un triángulo apoyado debajo del eje.

    El tamaño va en puntos y no en unidades de datos, para que no dependa de la escala del eje."""
    debajo = mtransforms.offset_copy(ax.transData, fig=ax.figure, x=0, y=-tamano * 0.55, units="points")
    ax.plot([x], [0], marker="^", markersize=tamano, linestyle="none",
            markerfacecolor=color if relleno else CREMA, markeredgecolor=color, markeredgewidth=1.8,
            transform=debajo, clip_on=False, zorder=5)


El relevamiento va adentro del notebook, como texto, así que no hace falta conexión. Son **45
visitas a 37 hogares**: ocho hogares recibieron una segunda visita. El apunte de la unidad 2 fija la
regla, la primera visita de cada hogar, y acá se usa la misma en las dos unidades. La copia
publicada está en
[`datos/relevamiento_plano.csv`](https://github.com/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-1-probabilidad-y-variables-aleatorias/datos/relevamiento_plano.csv).


In [ ]:
CSV = """id_hogar,localidad,provincia,fecha_visita,encuestador_legajo,encuestador_nombre,encuestador_telefono,personas,ingreso,cobertura_salud,satisfaccion,servicios
1,Ramallo,Buenos Aires,2025-03-14,E04,"Sosa, Diego",3407-419854,3,185000,si,buena,agua;luz;gas
1,Ramallo,Buenos Aires,2025-06-12,E04,"Sosa, Diego",3407-419854,4,201000,si,buena,agua;luz;gas
2,Córdoba,Córdoba,2025-03-15,E02,"Ledesma, Julio",351-4778120,5,240500,no,regular,agua;luz
3,Concepción,Tucumán,2025-03-15,E03,"Bianchi, Ana",381-4551907,2,,,,luz
4,Ramallo,Buenos Aires,2025-03-16,E01,"Rivas, Marta",3407-412233,4,198000,si,buena,agua;luz;gas
5,Ramallo,Buenos Aires,2025-03-16,E04,"Sosa, Diego",3407-419854,1,120000,,,luz
7,Córdoba,Córdoba,2025-03-17,E04,"Sosa, Diego",3407-419854,6,310000,no,mala,agua;luz;gas;cloacas
7,Córdoba,Córdoba,2025-06-13,E04,"Sosa, Diego",3407-419854,6,325000,no,regular,agua;luz;gas;cloacas
8,Concepción,Tucumán,2025-03-17,E03,"Bianchi, Ana",381-4551907,3,175000,si,regular,agua;luz
9,Ramallo,Buenos Aires,2025-03-18,E04,"Sosa, Diego",3407-419854,2,,no,mala,luz;gas
10,Córdoba,Córdoba,2025-03-19,E02,"Ledesma, Julio",351-4778120,4,205000,,,agua;luz;gas
11,Concepción,Tucumán,2025-03-20,E03,"Bianchi, Ana",381-4551907,7,260000,si,buena,agua;luz;gas;cloacas
11,Concepción,Tucumán,2025-06-13,E03,"Bianchi, Ana",381-4551907,7,268000,si,buena,agua;luz;gas;cloacas
13,Ramallo,Buenos Aires,2025-03-20,E04,"Sosa, Diego",3407-419854,3,190000,,,agua;luz
14,Córdoba,Córdoba,2025-03-21,E02,"Ledesma, Julio",351-4778120,2,168000,no,regular,agua
15,Ramallo,Buenos Aires,2025-03-21,E04,"Sosa, Diego",3407-419854,3,172000,si,regular,agua;luz;gas
16,Concepción,Tucumán,2025-03-24,E03,"Bianchi, Ana",381-4551907,4,195000,no,buena,agua;luz
17,Córdoba,Córdoba,2025-03-24,E04,"Sosa, Diego",3407-419854,2,210000,si,regular,agua;luz;gas
18,Córdoba,Córdoba,2025-03-25,E02,"Ledesma, Julio",351-4778120,5,155000,,,luz
19,Ramallo,Buenos Aires,2025-03-25,E04,"Sosa, Diego",3407-419854,3,188000,no,mala,agua;luz
20,Concepción,Tucumán,2025-03-26,E03,"Bianchi, Ana",381-4551907,6,225000,si,buena,agua;luz;gas;cloacas
20,Concepción,Tucumán,2025-06-16,E03,"Bianchi, Ana",381-4551907,5,231000,si,regular,agua;luz;gas;cloacas
21,Ramallo,Buenos Aires,2025-03-26,E04,"Sosa, Diego",3407-419854,2,163000,,,agua;luz
22,Córdoba,Córdoba,2025-03-27,E02,"Ledesma, Julio",351-4778120,4,201000,si,regular,agua;luz;gas
23,Ramallo,Buenos Aires,2025-03-27,E04,"Sosa, Diego",3407-419854,3,179000,no,buena,agua;luz
24,Concepción,Tucumán,2025-03-28,E03,"Bianchi, Ana",381-4551907,5,232000,si,regular,agua;luz;gas
25,Córdoba,Córdoba,2025-03-28,E04,"Sosa, Diego",3407-419854,1,148000,,,luz
26,Ramallo,Buenos Aires,2025-03-31,E01,"Rivas, Marta",3407-412233,4,216000,no,regular,agua;luz;gas
26,Ramallo,Buenos Aires,2025-06-17,E01,"Rivas, Marta",3407-412233,4,222000,no,buena,agua;luz;gas
27,Concepción,Tucumán,2025-03-31,E03,"Bianchi, Ana",381-4551907,2,193000,si,mala,agua;luz
30,Ramallo,Buenos Aires,2025-04-01,E01,"Rivas, Marta",3407-412233,7,244000,si,buena,agua;luz;gas;cloacas
30,Ramallo,Buenos Aires,2025-06-18,E01,"Rivas, Marta",3407-412233,8,259000,si,buena,agua;luz;gas;cloacas
31,Concepción,Tucumán,2025-04-02,E03,"Bianchi, Ana",381-4551907,4,181000,,,agua;luz
32,Córdoba,Córdoba,2025-04-02,E02,"Ledesma, Julio",351-4778120,3,207000,no,regular,agua;luz;gas
33,Ramallo,Buenos Aires,2025-04-02,E04,"Sosa, Diego",3407-419854,5,169000,si,regular,agua;luz
34,Concepción,Tucumán,2025-04-03,E03,"Bianchi, Ana",381-4551907,2,236000,no,buena,agua;luz;gas
35,Córdoba,Córdoba,2025-04-03,E04,"Sosa, Diego",3407-419854,4,152000,,,agua
36,Ramallo,Buenos Aires,2025-04-04,E01,"Rivas, Marta",3407-412233,6,199000,si,mala,agua;luz;gas
36,Ramallo,Buenos Aires,2025-06-19,E01,"Rivas, Marta",3407-412233,6,204000,si,regular,agua;luz;gas
37,Concepción,Tucumán,2025-04-04,E03,"Bianchi, Ana",381-4551907,3,223000,no,regular,agua;luz
38,Córdoba,Córdoba,2025-04-07,E02,"Ledesma, Julio",351-4778120,2,0,si,mala,luz
39,Ramallo,Buenos Aires,2025-04-07,E04,"Sosa, Diego",3407-419854,12,620000,no,buena,agua;luz;gas;cloacas
39,Ramallo,Buenos Aires,2025-06-20,E04,"Sosa, Diego",3407-419854,12,641000,no,regular,agua;luz;gas;cloacas
40,Concepción,Tucumán,2025-04-08,E03,"Bianchi, Ana",381-4551907,4,186000,,,agua;luz
42,Ramallo,Buenos Aires,2025-04-09,E01,"Rivas, Marta",3407-412233,2,176000,no,mala,agua;luz"""

visitas = pd.read_csv(StringIO(CSV), parse_dates=["fecha_visita"])
primera = (visitas.sort_values("fecha_visita").drop_duplicates("id_hogar", keep="first")
           .sort_values("id_hogar").reset_index(drop=True))
segunda = (visitas.sort_values("fecha_visita").drop_duplicates("id_hogar", keep="last")
           .sort_values("id_hogar").reset_index(drop=True))
hogares = primera   # la regla del apunte: un hogar, su primera visita

print(f"{len(visitas)} visitas, {len(hogares)} hogares, {hogares['localidad'].nunique()} localidades")
print("personas por hogar, primera visita:",
      hogares["personas"].value_counts().sort_index().to_dict())


---
# 1. La distribución de una variable discreta

*Apunte, secciones 2.1, 3.1 a 3.3 y 3.6.*

Del bolillero se sacan bolillas sin reposición, y la variable `X` es la cantidad de blancas que
salen. La **función de probabilidad** asigna a cada valor posible su probabilidad, y se arma
recorriendo el árbol: cada camino es una compuesta, y los caminos que dan la misma cantidad de
blancas se suman. La **acumulada** `F(x)` suma las probabilidades de izquierda a derecha: nunca
baja y termina en 1.

El apunte arma la tabla con dos extracciones: `6/56`, `30/56` y `20/56`. Acá se puede pedir más, y
se ve por qué la sección 3.6 dice que contar a mano no alcanza: con seis extracciones el árbol tiene
hasta 64 caminos. La última columna de la tabla compara el conteo con una fórmula cerrada que
`scipy` ya trae. Esas distribuciones con nombre son tema del bloque 2 de la asignatura; acá solo se
comprueba que dan lo mismo que el árbol.


In [ ]:
#@title Cantidad de blancas: P(X = x) y F(x) { run: "auto", display-mode: "form" }

blancas = 5  #@param {type:"slider", min:1, max:12, step:1}
negras = 3  #@param {type:"slider", min:1, max:12, step:1}
extracciones = 2  #@param {type:"slider", min:1, max:6, step:1}
reposicion = False  #@param {type:"boolean"}

from math import perm

total = blancas + negras
k = extracciones if reposicion else min(extracciones, total)
den = total ** k if reposicion else perm(total, k)   # el denominador común de todos los caminos


def caminos(b, n, faltan):
    """Recorre el árbol y devuelve (cantidad de blancas, casos favorables sobre den) de cada camino."""
    if faltan == 0:
        return [(0, 1)]
    salida = []
    if b > 0:
        for x, casos in caminos(b if reposicion else b - 1, n, faltan - 1):
            salida.append((x + 1, casos * b))
    if n > 0:
        for x, casos in caminos(b, n if reposicion else n - 1, faltan - 1):
            salida.append((x, casos * n))
    return salida


todos = caminos(blancas, negras, k)
valores = np.arange(0, k + 1)
numeradores = np.array([sum(c for x, c in todos if x == v) for v in valores])
probabilidad = numeradores / den
acumulada = np.cumsum(probabilidad)
if reposicion:
    con_nombre = stats.binom(k, blancas / total).pmf(valores)
    nombre = "fórmula binomial"
else:
    con_nombre = stats.hypergeom(M=total, n=blancas, N=k).pmf(valores)
    nombre = "fórmula hipergeométrica"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax1.bar(valores, probabilidad, color=TEAL, width=0.6)
for v, p, nmr in zip(valores, probabilidad, numeradores):
    ax1.text(v, p + 0.01, f"{nmr}/{den}", ha="center", fontsize=9)
ax1.set_xticks(valores); ax1.set_ylim(0, max(probabilidad) * 1.2)
ax1.set_xlabel("x: cantidad de blancas"); ax1.set_ylabel("P(X = x)")
ax1.set_title(f"Función de probabilidad ({len(todos)} caminos del árbol)")
ax2.step(np.append(valores, k + 1), np.append(acumulada, 1), where="post", color=CORAL, linewidth=2.2)
ax2.plot(valores, acumulada, "o", color=CORAL)
ax2.axhline(1, color=TINTA, linestyle=":", linewidth=1)
ax2.set_xticks(valores); ax2.set_ylim(0, 1.08); ax2.set_xlim(-0.5, k + 1)
ax2.set_xlabel("x"); ax2.set_ylabel("F(x) = P(X ≤ x)")
ax2.set_title("Acumulada: nunca baja, termina en 1")
plt.tight_layout(); plt.show()

if k < extracciones:
    print(f"No se pueden sacar {extracciones} bolillas de {total} sin reposición: se usan {k}.\n")
tabla = pd.DataFrame({"x": valores,
                      "P(X = x), por el árbol": [f"{nmr}/{den}" for nmr in numeradores],
                      "decimal": [num(p) for p in probabilidad],
                      "F(x)": [num(a) for a in acumulada],
                      nombre: [num(p) for p in con_nombre]}).set_index("x")
display(tabla)
print(f"Suma de las probabilidades: {num(probabilidad.sum())}.  "
      + ("La fórmula cerrada da exactamente lo mismo que el árbol."
         if np.abs(probabilidad - con_nombre).max() < 1e-9 else "Atención: la fórmula no coincide."))


---
# 2. La variable continua: la probabilidad es un área

*Apunte, secciones 2.3, 2.4 y 3.4.*

Una variable continua puede tomar cualquier valor de un intervalo, y ahí la tabla de la sección 1
no sirve: hay infinitos valores. Se la describe con una **función de densidad**, y la probabilidad
de un intervalo es el **área** bajo la curva sobre ese intervalo. El área total vale 1.

La consecuencia que cuesta al principio: la probabilidad de un valor exacto es cero, porque un
punto no tiene ancho. A la derecha, el intervalo alrededor de un valor se achica con el deslizador,
y el área se va a cero con él.

El ejemplo es el tiempo que tarda un trámite, en minutos, con una forma de campana.


In [ ]:
#@title Área bajo la densidad { run: "auto", display-mode: "form" }

media = 30  #@param {type:"slider", min:10, max:60, step:1}
desvio = 8  #@param {type:"slider", min:2, max:20, step:1}
desde = 20  #@param {type:"slider", min:0, max:90, step:1}
hasta = 40  #@param {type:"slider", min:0, max:90, step:1}
valor_exacto = 30  #@param {type:"slider", min:0, max:90, step:1}
ancho_alrededor = 2.0  #@param {type:"slider", min:0.01, max:10, step:0.01}

a, b = sorted([desde, hasta])
X = stats.norm(media, desvio)
izquierda, derecha = max(0, media - 4 * desvio), media + 4 * desvio
xs = np.linspace(izquierda, derecha, 600)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4), sharey=True)
for ax in (ax1, ax2):
    ax.plot(xs, X.pdf(xs), color=TINTA, linewidth=1.8)
    ax.set_xlabel("minutos"); ax.set_xlim(izquierda, derecha); ax.set_ylim(0, X.pdf(media) * 1.15)
    ax.set_yticks([])
zona = (xs >= a) & (xs <= b)
ax1.fill_between(xs[zona], X.pdf(xs[zona]), color=TEAL, alpha=0.75)
area = X.cdf(b) - X.cdf(a)
ax1.set_title(f"P({a} ≤ X ≤ {b}) = área = {num(area, 4)}")
lo, hi = valor_exacto - ancho_alrededor / 2, valor_exacto + ancho_alrededor / 2
franja = np.linspace(lo, hi, 50)
ax2.fill_between(franja, X.pdf(franja), color=CORAL, alpha=0.85)
ax2.axvline(valor_exacto, color=CORAL, linewidth=1.2, linestyle="--")
area2 = X.cdf(hi) - X.cdf(lo)
ax2.set_title(f"P({num(lo, 2)} ≤ X ≤ {num(hi, 2)}) = {num(area2, 4)}")
plt.tight_layout(); plt.show()

print(f"Área total bajo la curva: {num(X.cdf(np.inf) - X.cdf(-np.inf), 4)}")
print(f"P({a} ≤ X ≤ {b}) = {num(area, 4)}"
      + ("   (el intervalo cae fuera de lo que muestra el gráfico)" if b < izquierda or a > derecha else ""))
print(f"P(X exactamente = {valor_exacto}) = 0. Un intervalo de ancho {num(ancho_alrededor, 2)} alrededor tiene "
      f"probabilidad {num(area2, 4)}, y se achica con el ancho.")


---
# 3. La distribución de las personas por hogar

*Apunte, sección 3.5.*

Ahora con datos reales. El experimento es elegir un hogar al azar entre los 37, y `X` es la
cantidad de personas que viven en él. Como cada hogar tiene la misma chance de salir, la
probabilidad de cada valor es su frecuencia relativa: el enfoque empírico de la unidad 1, usado
como distribución.

El apunte toma la **primera visita** de cada hogar. Ocho hogares tuvieron una segunda visita, y en
tres de ellos la cantidad de personas cambió. Cambiá la visita y mirá el hueco entre 7 y 12: con la
segunda visita aparece un hogar de 8 y el hueco se achica. La regla que se elige hay que declararla,
como el denominador de la unidad 1.


In [ ]:
#@title Personas por hogar: P(X = x) y F(x) { run: "auto", display-mode: "form" }

visita = "primera"  #@param ["primera", "segunda"]

datos = primera if visita == "primera" else segunda
x, p, cuenta = distribucion_empirica(datos["personas"])
F = np.cumsum(p)
m = medidas(x, p)
mediana = x[np.searchsorted(F, 0.5)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax1.bar(x, p, color=TEAL, width=0.7)
for xi, pi, ci in zip(x, p, cuenta):
    ax1.text(xi, pi + 0.005, f"{ci}", ha="center", fontsize=10)
ax1.set_xticks(range(1, int(x.max()) + 1)); ax1.set_ylim(0, p.max() * 1.25)
ax1.set_xlabel("x: personas en el hogar"); ax1.set_ylabel("P(X = x) = hogares / 37")
ax1.set_title(f"{visita.capitalize()} visita: {len(datos)} hogares, {int(datos['personas'].sum())} personas")
huecos = sorted(set(range(1, int(x.max()))) - set(x))
if huecos:
    ax1.axvspan(min(huecos) - 0.5, max(huecos) + 0.5, color=MOSTAZA, alpha=0.18)
    ax1.text((min(huecos) + max(huecos)) / 2, p.max() * 1.1, "sin hogares", ha="center", fontsize=9)
ax2.step(np.append(x, x.max() + 1), np.append(F, 1), where="post", color=CORAL, linewidth=2.2)
ax2.plot(x, F, "o", color=CORAL)
ax2.axhline(0.5, color=TINTA, linestyle=":", linewidth=1)
ax2.annotate(f"mediana = {mediana}", xy=(mediana, 0.5), xytext=(mediana + 2.5, 0.42), fontsize=10,
             arrowprops=dict(arrowstyle="->", color=TINTA))
ax2.set_xticks(range(1, int(x.max()) + 2)); ax2.set_ylim(0, 1.08)
ax2.set_xlabel("x"); ax2.set_ylabel("F(x) = P(X ≤ x)")
ax2.set_title("Acumulada: donde cruza 0,5 está la mediana")
plt.tight_layout(); plt.show()

tabla = pd.DataFrame({"x": x, "hogares": cuenta, "P(X = x)": [num(v) for v in p],
                      "F(x)": [num(v) for v in F]}).set_index("x")
display(tabla)
print(f"E(X) = {num(m['E(X)'])} personas   desvío standard = {num(m['desvío standard'])}   "
      f"α₃ = {num(m['asimetría α₃'])}   α₄ = {num(m['kurtosis α₄'])}")


---
# 4. La esperanza es el punto de equilibrio

*Apunte, secciones 4.1, 4.2 y 4.5.*

La **esperanza** `E(X)` es el promedio de los valores, cada uno pesado por su probabilidad. Si se
dibujan las probabilidades como pesos sobre una barra, la esperanza es el punto donde la barra
queda en equilibrio. No tiene por qué ser un valor posible: 3,78 personas no existe, y aun así es
el centro de la distribución.

Los pesos de la izquierda son los 36 hogares de 1 a 7 personas del relevamiento, y su equilibrio
solo, sin nada más, es el triángulo hueco: `3,5556`. El valor lejano representa al hogar de 12.
Corré ese valor y cambiá cuántos hogares lo tienen: un peso chico, muy lejos, mueve el apoyo igual.
Con un hogar de 12 el apoyo se corre `0,23`, que es lo que dice el apunte.


In [ ]:
#@title La balanza de la esperanza { run: "auto", display-mode: "form" }

valor_lejano = 12  #@param {type:"slider", min:8, max:30, step:1}
hogares_con_ese_valor = 1  #@param {type:"slider", min:0, max:12, step:1}

base_x, _, base_cuenta = distribucion_empirica(hogares.loc[hogares["personas"] < 8, "personas"])
total = base_cuenta.sum() + hogares_con_ese_valor
x = np.append(base_x, valor_lejano)
p = np.append(base_cuenta / total, hogares_con_ese_valor / total)
mu = (x * p).sum()
mu_base = (base_x * base_cuenta).sum() / base_cuenta.sum()   # los 36 hogares solos, siempre 3,5556

fig, ax = plt.subplots(figsize=(11, 4.4))
ax.bar(x[:-1], p[:-1], color=TEAL, width=0.6)
ax.bar([x[-1]], [p[-1]], color=CORAL, width=0.6)
ax.plot([0.4, max(valor_lejano, 8) + 0.6], [0, 0], color=TINTA, linewidth=3)
dibujar_fulcro(ax, mu_base, color=TINTA, relleno=False)
dibujar_fulcro(ax, mu, color=CORAL, relleno=True)
ax.set_xlim(0, max(valor_lejano, 8) + 1); ax.set_ylim(-0.02, max(p.max(), 0.05) * 1.25)
ax.set_xticks(range(1, max(valor_lejano, 8) + 1)); ax.grid(False)
ax.set_xlabel("personas en el hogar"); ax.set_ylabel("probabilidad")
ax.set_title(f"{int(total)} hogares: E(X) = {num(mu, 4)}. Los 36 de la izquierda solos: {num(mu_base, 4)}")
plt.show()

print("E(X) = Σ x · P(X = x)")
print("     = " + " + ".join(f"{int(xi)}·{num(pi, 3)}" for xi, pi in zip(x, p)))
print(f"     = {num(mu, 4)}")
if hogares_con_ese_valor:
    print(f"El valor lejano aporta {int(valor_lejano)} × {num(p[-1], 4)} = {num(valor_lejano * p[-1], 4)} a la suma, "
          f"y corre el apoyo de {num(mu_base, 4)} a {num(mu, 4)}: {num(mu - mu_base, 4)} personas.")
else:
    print("Sin ningún hogar en el valor lejano, el apoyo queda en el equilibrio de los 36: los dos triángulos coinciden.")


---
# 5. Una función de la variable, y las propiedades

*Apunte, secciones 4.3, 4.4, 5.2, 5.3, 5.5 y 6.2.*

Si `X` son las personas por hogar, `Y = a·X + b` es otra variable aleatoria. Sus medidas no hace
falta recalcularlas: `E(Y) = a·E(X) + b`, y `V(Y) = a²·V(X)`. Sumar una constante mueve el centro y
no toca la dispersión. Multiplicar por una constante multiplica el desvío por `|a|` y la varianza
por `a²`.

Qué mover y qué mirar. Con `a = −2`, la varianza sigue positiva y `α₃` cambia de signo: la cola se
dio vuelta. Con `b = −3,5`, la media queda cerca de cero y el CV explota: es la primera condición de
la sección 6.2, la media tiene que estar lejos del cero. Y `α₄` no cambia nunca: la forma de las
colas no depende de la escala. La tabla también verifica la fórmula corta
`V(X) = E(X²) − [E(X)]²`, con los corchetes que el apunte remarca.


In [ ]:
#@title Y = a·X + b { run: "auto", display-mode: "form" }

a = 2.0  #@param {type:"slider", min:-3, max:3, step:0.5}
b = 1.0  #@param {type:"slider", min:-10, max:10, step:0.5}

x, p, _ = distribucion_empirica(hogares["personas"])
y = a * x + b
mX, mY = medidas(x, p), medidas(y, p)
e_x2 = (x ** 2 * p).sum()

fig, ax = plt.subplots(figsize=(11.5, 4.4))
ancho = min(0.6, 0.6 * abs(a)) if a else 0.6
ax.bar(x, p, color=TEAL, width=0.6, alpha=0.9, label="X: personas por hogar")
ax.bar(y, p, color=CORAL, width=ancho, alpha=0.75, label=f"Y = {num(a, 1)}·X + {num(b, 1)}")
piso = min(0, y.min()) - 1
techo = max(x.max(), y.max()) + 1
ax.plot([piso, techo], [0, 0], color=TINTA, linewidth=2)
dibujar_fulcro(ax, mX["E(X)"], color=TEAL)
dibujar_fulcro(ax, mY["E(X)"], color=CORAL)
ax.set_xlim(piso, techo); ax.set_ylim(-0.02, p.max() * 1.25); ax.grid(False)
ax.legend(loc="upper right"); ax.set_ylabel("probabilidad")
ax.set_title(f"E(Y) = {num(mY['E(X)'])}   desvío standard de Y = {num(mY['desvío standard'])}")
plt.show()

formulas = {"E(X)": a * mX["E(X)"] + b, "V(X)": a ** 2 * mX["V(X)"],
            "desvío standard": abs(a) * mX["desvío standard"],
            "CV (%)": np.nan, "asimetría α₃": np.sign(a) * mX["asimetría α₃"] if a else np.nan,
            "kurtosis α₄": mX["kurtosis α₄"] if a else np.nan}
comparacion = tabla_medidas(("X", mX), ("Y, calculada directo", mY), ("Y, por las fórmulas", formulas))
comparacion.loc["CV (%)", "Y, por las fórmulas"] = "no hay fórmula: depende de b"
display(comparacion)
print(f"Fórmula corta: E(X²) − [E(X)]² = {num(e_x2)} − {num(mX['E(X)'] ** 2)} = {num(e_x2 - mX['E(X)'] ** 2)} = V(X)")
if a < 0:
    print("Con a negativo la varianza igual es positiva, el desvío usa |a|, y la asimetría cambia de signo.")
if a == 0:
    print("Con a = 0, Y es una constante: su varianza es 0 y las medidas de forma no existen.")
if a and abs(mY["E(X)"]) < 0.5:
    print(f"La media de Y quedó en {num(mY['E(X)'], 2)}, casi cero: el CV no se puede usar (sección 6.2).")


---
# 6. Misma media, distinta dispersión, y la estandarización

*Apunte, secciones 5.1, 5.4 y 5.7.*

Dos distribuciones pueden tener la misma esperanza y comportarse muy distinto: el apunte abre la
sección 5 con un juego que paga siempre 1.000 y otro que paga 0 o 2.000. El **desvío standard** mide
cuánto se alejan los valores del centro, en las unidades de la variable. La **estandarización**
`Z = (X − μ) / σ` resta el centro y divide por el desvío: toda variable estandarizada tiene media 0
y desvío 1, y por eso permite comparar cosas medidas en escalas distintas.

A la izquierda, dos distribuciones con el mismo punto de apoyo y distinto ancho. A la derecha, las
mismas dos en `Z`: la diferencia de escala desaparece. Mové el desvío de cada una.


In [ ]:
#@title Dispersión y variable estandarizada { run: "auto", display-mode: "form" }

media = 10  #@param {type:"slider", min:5, max:20, step:1}
desvio_1 = 1.0  #@param {type:"slider", min:0.5, max:5, step:0.5}
desvio_2 = 3.0  #@param {type:"slider", min:0.5, max:5, step:0.5}

soporte = np.arange(-30, 61)   # ancho de sobra, para que ninguna campana quede recortada


def discreta(mu, sigma):
    p = stats.norm(mu, sigma).pdf(soporte)
    return p / p.sum()


p1, p2 = discreta(media, desvio_1), discreta(media, desvio_2)
m1, m2 = medidas(soporte, p1), medidas(soporte, p2)
mayor = max(desvio_1, desvio_2)
lim = (media - 4 * mayor - 0.5, media + 4 * mayor + 0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))
ax1.bar(soporte - 0.2, p1, width=0.4, color=TEAL, label=f"desvío {num(desvio_1, 1)}")
ax1.bar(soporte + 0.2, p2, width=0.4, color=CORAL, label=f"desvío {num(desvio_2, 1)}")
ax1.plot(lim, [0, 0], color=TINTA, linewidth=2)
dibujar_fulcro(ax1, m1["E(X)"], color=TEAL, tamano=18)
dibujar_fulcro(ax1, m2["E(X)"], color=CORAL, tamano=12)
ax1.set_xlim(*lim); ax1.set_ylim(-0.03, max(p1.max(), p2.max()) * 1.2); ax1.grid(False)
ax1.legend(loc="upper right"); ax1.set_title(f"Mismo apoyo en E(X) = {num(media, 0)}, distinto ancho")
for p, m, color, corr in [(p1, m1, TEAL, -0.08), (p2, m2, CORAL, 0.08)]:
    z = (soporte - m["E(X)"]) / m["desvío standard"]
    ax2.bar(z + corr, p / p.max(), width=0.16, color=color, alpha=0.85)
ax2.set_xlim(-4, 4); ax2.set_yticks([]); ax2.grid(False)
ax2.set_xlabel("z: desvíos respecto de la media")
ax2.set_title("Las mismas dos, estandarizadas")
fig.subplots_adjust(wspace=0.15, bottom=0.18); plt.show()

display(tabla_medidas((f"desvío {num(desvio_1, 1)}", m1), (f"desvío {num(desvio_2, 1)}", m2))
        .loc[["E(X)", "V(X)", "desvío standard"]])
for etiqueta, p, m in [(f"desvío {num(desvio_1, 1)}", p1, m1), (f"desvío {num(desvio_2, 1)}", p2, m2)]:
    z = (soporte - m["E(X)"]) / m["desvío standard"]
    mz = medidas(z, p)
    print(f"Z de la de {etiqueta}: E(Z) = {num(mz['E(X)'], 4)}   V(Z) = {num(mz['V(X)'], 4)}")


---
# 7. El coeficiente de variabilidad

*Apunte, secciones 6.1 y 6.2.*

El desvío tiene las unidades de la variable, así que no sirve para comparar la dispersión de las
personas por hogar con la del ingreso. El **coeficiente de variabilidad** divide el desvío por la
media y da un número sin unidades, que se expresa en porcentaje.

Tiene una condición: la variable tiene que estar en **escala de razón**, con un cero que signifique
"nada". La temperatura no cumple. Las dos últimas barras son las mismas mediciones, en grados
centígrados y en Fahrenheit: el desvío se convierte bien, pero el CV cambia, porque el cero de la
escala se movió. Con los números del apunte, `25 %` pasa a `13 %`. Las personas y el ingreso no
tienen ese problema.


In [ ]:
#@title CV de tres variables, y una que no lo admite { run: "auto", display-mode: "form" }

temperaturas = "media 20 y desvío 5, como el apunte"  #@param ["media 20 y desvío 5, como el apunte", "doce meses de una ciudad"]

if temperaturas.startswith("media 20"):
    grados_c = np.array([15.0, 25.0] * 6)
else:
    grados_c = np.array([23, 22, 20, 17, 13, 10, 9, 11, 13, 15, 18, 21], dtype=float)
grados_f = grados_c * 9 / 5 + 32

series = {
    "personas por hogar": hogares["personas"].astype(float).to_numpy(),
    "ingreso mensual ($)": hogares["ingreso"].dropna().to_numpy(),
    "temperatura (°C)": grados_c,
    "temperatura (°F)": grados_f,
}
resumen = pd.DataFrame({
    nombre: {"media": v.mean(), "desvío standard": v.std(ddof=0), "CV (%)": 100 * v.std(ddof=0) / v.mean()}
    for nombre, v in series.items()
}).T

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.barh(resumen.index, resumen["CV (%)"], color=[TEAL, TEAL, CORAL, CORAL], height=0.55)
for i, v in enumerate(resumen["CV (%)"]):
    ax.text(v + 1, i, f"{num(v, 1)} %", va="center")
ax.set_xlim(0, resumen["CV (%)"].max() * 1.25); ax.invert_yaxis()
ax.set_xlabel("coeficiente de variabilidad (%)")
ax.set_title("El mismo clima da dos CV: la temperatura no admite el coeficiente")
plt.show()

display(resumen.map(lambda v: num(v, 2)))
print(f"Temperatura: el desvío pasa de {num(grados_c.std(), 2)} °C a {num(grados_f.std(), 2)} °F, que es lo mismo × 9/5.")
print(f"El CV pasa de {num(resumen.loc['temperatura (°C)', 'CV (%)'], 1)} % a "
      f"{num(resumen.loc['temperatura (°F)', 'CV (%)'], 1)} %, porque la media cambió de origen: el cero es una convención.")
print("Personas e ingreso tienen un cero real, y sus CV sí se pueden comparar.")


---
# 8. La asimetría

*Apunte, secciones 7.1 a 7.3.*

El **coeficiente de asimetría** `α₃` dice hacia qué lado se estira la distribución. Es el promedio
del cubo de los valores estandarizados: conserva el signo, así que las distancias hacia la derecha
suman y las de la izquierda restan. Vale cero si los dos lados se equilibran.

Debajo del eje van las tres medidas de posición con los símbolos de la ilustración de la sección
7.2: el círculo coral es el **modo**, el triángulo mostaza la **mediana** y el rombo teal la
**media**. Con la cola a la derecha, la media es la que más se corre hacia la cola, y por eso una
distribución asimétrica se informa con la mediana. Mové el sesgo de negativo a positivo y mirá cómo
se ordenan. Cerca de cero las tres coinciden, y el apunte avisa que ese orden es una guía, no un
teorema.


In [ ]:
#@title Cola larga y las tres medidas { run: "auto", display-mode: "form" }

sesgo = 6.0  #@param {type:"slider", min:-10, max:10, step:0.5}

soporte = np.arange(0, 24.01, 0.5)
p = stats.skewnorm(sesgo, loc=12, scale=4).pdf(soporte)
p = p / p.sum()
m = medidas(soporte, p)
F = np.cumsum(p)
modo = soporte[np.argmax(p)]
mediana = soporte[np.searchsorted(F, 0.5)]
media = m["E(X)"]
a3 = m["asimetría α₃"]

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.bar(soporte, p, color=TEAL_PALIDO, edgecolor=TEAL, width=0.42)
ax.plot([-0.5, 24.5], [0, 0], color=TINTA, linewidth=2)
paso = -p.max() * 0.11
for nivel, (valor, marca, color, etiqueta) in enumerate([
        (modo, "o", CORAL, f"modo = {num(modo, 1)}"), (mediana, "^", MOSTAZA, f"mediana = {num(mediana, 1)}"),
        (media, "D", TEAL, f"media = {num(media, 2)}")], start=1):
    ax.plot([valor, valor], [0, paso * nivel], color=color, linewidth=1, linestyle=":", clip_on=False)
    ax.plot(valor, paso * nivel, marca, color=color, markersize=12, clip_on=False, label=etiqueta)
ax.set_xlim(-0.5, 24.5); ax.set_ylim(paso * 4, p.max() * 1.2); ax.grid(False)
ax.set_yticks([]); ax.legend(loc="upper right")
if a3 > 0.05:
    lectura = "la cola se estira hacia la derecha"
elif a3 < -0.05:
    lectura = "la cola se estira hacia la izquierda"
else:
    lectura = "no hay una cola dominante"
ax.set_title(f"α₃ = {num(a3, 4)}: {lectura}")
plt.show()

orden = sorted([("modo", modo), ("mediana", mediana), ("media", media)], key=lambda t: t[1])
texto = orden[0][0]
for (n1, v1), (n2, v2) in zip(orden, orden[1:]):
    texto += (" = " if abs(v2 - v1) < 1e-9 else " < ") + n2
print("Orden de izquierda a derecha: " + texto)
print(f"α₃ = {num(a3, 4)}   (positiva: cola a la derecha; negativa: cola a la izquierda; cero: simétrica)")
x_h, p_h, _ = distribucion_empirica(hogares["personas"])
print(f"Para comparar, las personas por hogar del relevamiento tienen α₃ = {num(medidas(x_h, p_h)['asimetría α₃'], 4)}")


---
# 9. La kurtosis

*Apunte, secciones 8.1 a 8.3.*

La **kurtosis** `α₄` usa la cuarta potencia, así que los valores más lejanos pesan mucho más que los
cercanos. Mide cuánto pesan las **colas**, no qué tan alto es el pico. La referencia es la
distribución normal, con `α₄ = 3`: por encima es leptocúrtica, por debajo platicúrtica.

Los tres perfiles tienen la misma varianza, y por eso se comparan bien: uno con colas que se
apagan enseguida, la normal y uno con colas que se resisten. El deslizador carga las colas del
tercero: más valor, más peso en los extremos. La zona sombreada marca las colas, a más de dos
desvíos del centro, y la leyenda dice cuánta probabilidad tiene cada perfil ahí: eso es lo que hay
que mirar, no el pico. Abajo, las dos convenciones sobre las personas por hogar: la kurtosis y el
exceso, que le resta 3.

El perfil de colas pesadas es una distribución con nombre que la asignatura ve más adelante, la t
de Student. Acá solo importa su forma.


In [ ]:
#@title Colas livianas, normales y pesadas { run: "auto", display-mode: "form" }

colas_mas_pesadas = 8  #@param {type:"slider", min:1, max:10, step:1}

gl = 45 - 4 * colas_mas_pesadas   # de 41 (casi normal) a 5 (colas muy pesadas)
xs = np.linspace(-5, 5, 801)
perfiles = {
    "colas livianas": stats.uniform(-np.sqrt(3), 2 * np.sqrt(3)),
    "normal": stats.norm(0, 1),
    "colas pesadas": stats.t(gl, scale=np.sqrt((gl - 2) / gl)),
}
kurtosis = {"colas livianas": 1.8, "normal": 3.0, "colas pesadas": 3 + 6 / (gl - 4)}
en_colas = {nombre: 2 * X.cdf(-2) for nombre, X in perfiles.items()}

fig, ax = plt.subplots(figsize=(11, 4.6))
for (nombre, X), color in zip(perfiles.items(), [MOSTAZA, TINTA, CORAL]):
    ax.plot(xs, X.pdf(xs), color=color, linewidth=2,
            label=f"{nombre}: α₄ = {num(kurtosis[nombre], 2)}, en las colas {num(100 * en_colas[nombre], 1)} %")
ax.axvspan(-5, -2, color=TEAL_PALIDO, alpha=0.6); ax.axvspan(2, 5, color=TEAL_PALIDO, alpha=0.6)
ax.set_xlim(-5, 5); ax.set_ylim(0, 0.62); ax.set_yticks([]); ax.grid(False)
ax.set_xlabel("desvíos respecto de la media (las tres tienen varianza 1)")
ax.legend(loc="upper right"); ax.set_title("La kurtosis se lee en las colas, no en el pico")
plt.show()

tabla = pd.DataFrame({
    "kurtosis α₄": {k: num(v, 4) for k, v in kurtosis.items()},
    "exceso (α₄ − 3)": {k: num(v - 3, 4) for k, v in kurtosis.items()},
    "P(|Z| > 2)": {k: num(100 * v, 2) + " %" for k, v in en_colas.items()},
    "categoría": {k: "platicúrtica" if v < 3 else "mesocúrtica" if v == 3 else "leptocúrtica"
                  for k, v in kurtosis.items()},
})
display(tabla)

personas = hogares["personas"]
print("Las personas por hogar, con las convenciones de la sección 8.3:")
print(f"  scipy.stats.kurtosis(x, fisher=False) = {num(stats.kurtosis(personas, fisher=False), 4)}   ← α₄, distribución completa")
print(f"  scipy.stats.kurtosis(x)               = {num(stats.kurtosis(personas), 4)}   ← exceso, distribución completa")
print(f"  pandas: x.kurt()                      = {num(personas.kurt(), 4)}   ← exceso, corregido por muestra")
print(f"  scipy.stats.skew(x)                   = {num(stats.skew(personas), 4)}   ← α₃, distribución completa")
print(f"  pandas: x.skew()                      = {num(personas.skew(), 4)}   ← α₃, corregido por muestra")


---
# 10. Un hogar de 37, y las seis medidas juntas

*Apunte, secciones 8.4, 9 y 10.*

El hogar de 12 personas es uno de 37. Sacarlo mueve poco la media, bastante el desvío, y muchísimo
la asimetría y la kurtosis, porque esas dos usan potencias altas y un valor lejano las gobierna: a
mayor potencia, más manda el valor más lejano. La kurtosis pasa de leptocúrtica a platicúrtica con
un solo hogar.

Esto no dice que haya que descartarlo: el T8001 lo revisó y lo conservó. Dice que el mismo conjunto
de datos admite dos lecturas, y que hay que informar cuál se usó. La tabla muestra siempre las dos,
como la de la sección 8.4 del apunte; la casilla elige cuál se dibuja.


In [ ]:
#@title Las seis medidas, con y sin el hogar de 12 { run: "auto", display-mode: "form" }

incluir_el_hogar_de_12 = True  #@param {type:"boolean"}

todos = hogares["personas"]
sin_12 = todos[todos < 12]
m_con = medidas(*distribucion_empirica(todos)[:2])
m_sin = medidas(*distribucion_empirica(sin_12)[:2])
elegida = todos if incluir_el_hogar_de_12 else sin_12
m_elegida = m_con if incluir_el_hogar_de_12 else m_sin

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
x, p, cuenta = distribucion_empirica(elegida)
ax1.bar(x, p, color=[CORAL if v == 12 else TEAL for v in x], width=0.7)
for xi, pi, ci in zip(x, p, cuenta):
    ax1.text(xi, pi + 0.005, str(ci), ha="center", fontsize=9)
ax1.plot([0.5, 12.5], [0, 0], color=TINTA, linewidth=2)
dibujar_fulcro(ax1, m_elegida["E(X)"], color=CORAL)
ax1.set_xlim(0.5, 12.5); ax1.set_xticks(range(1, 13)); ax1.set_ylim(-0.02, 0.32); ax1.grid(False)
ax1.set_xlabel("personas en el hogar"); ax1.set_ylabel("probabilidad")
ax1.set_title(f"{len(elegida)} hogares: E(X) = {num(m_elegida['E(X)'])}, α₄ = {num(m_elegida['kurtosis α₄'], 2)}")

nombres = list(m_con)
cambio = [100 * (m_sin[k] - m_con[k]) / m_con[k] for k in nombres]
de_forma = [k.startswith(("asimetría", "kurtosis")) for k in nombres]
ax2.barh(nombres, cambio, color=[CORAL if f else TEAL for f in de_forma], height=0.55)
for i, c in enumerate(cambio):
    ax2.text(c + (1.5 if c >= 0 else -1.5), i, f"{num(c, 1)} %", va="center",
             ha="left" if c >= 0 else "right", fontsize=9)
ax2.axvline(0, color=TINTA, linewidth=1); ax2.invert_yaxis()
ax2.set_xlim(min(cambio) - 25, max(cambio) + 25)
ax2.set_xlabel("cambio relativo al sacar el hogar de 12 (%)")
ax2.set_title("Las dos de forma cambian de lectura")
fig.subplots_adjust(wspace=0.45, bottom=0.18); plt.show()

comparacion = tabla_medidas(("con los 37 hogares", m_con), ("sin el hogar de 12", m_sin))
lectura = []
for k, c in zip(nombres, cambio):
    if k == "CV (%)":
        lectura.append(f"{num(m_sin[k] - m_con[k], 1)} puntos")
    elif k.startswith("kurtosis"):
        lectura.append(("leptocúrtica" if m_con[k] > 3 else "platicúrtica") + " → " +
                       ("leptocúrtica" if m_sin[k] > 3 else "platicúrtica"))
    else:
        lectura.append(f"{num(c, 1)} %")
comparacion["cambio"] = lectura
display(comparacion)
print("Las potencias mandan: la media usa la primera, la varianza la segunda, la asimetría la tercera y la kurtosis la cuarta.")
print("Cuanto más alta la potencia, más pesa el hogar de 12 en la medida.")


---
## Cierre

Diez formularios, y en todos se puede volver a cambiar algo:

1. La distribución del bolillero armada por el árbol, sobre 56, y la comprobación con `scipy`.
2. La variable continua: la probabilidad como área, y el valor exacto con probabilidad cero.
3. Las personas por hogar, la regla de la primera visita y la mediana en la acumulada.
4. La esperanza como punto de equilibrio, y el peso de un valor lejano.
5. `Y = a·X + b`: las seis medidas de `Y`, y qué cambia con `a`, con `b` y con el signo.
6. Misma media, distinta dispersión, y la estandarización.
7. El coeficiente de variabilidad, y la escala que no lo admite.
8. La asimetría, con el modo, la mediana y la media.
9. La kurtosis en las colas, y las dos convenciones.
10. Las seis medidas con y sin el hogar de 12.

Quedan afuera, porque se entienden leyendo, la lectura de largo plazo de la esperanza de la sección
4.1, la varianza de una suma de la sección 5.6 y los momentos de la sección 9.

> **Apunte de la unidad 2:**
> [E8102_Apunte_U2_Variable_Aleatoria.pdf](https://github.com/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-1-probabilidad-y-variables-aleatorias/E8102_Apunte_U2_Variable_Aleatoria.pdf)
